In [8]:
import pdfplumber
import csv
import os
from pathlib import Path
import pandas as pd

def extract_pdf_to_rows(pdf_path: str) -> list[list]:
    """Извлекает все строки таблиц из PDF-выписки Halyk Bank."""
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    cleaned = [
                        (cell.replace("\n", " ").strip() if cell else "")
                        for cell in row
                    ]
                    rows.append(cleaned)
                rows.append([])  # разделитель между таблицами
    return rows


def extract_all_pdfs(pdf_paths: list[str], output_dir: str = ".") -> list[str]:
    """Обрабатывает список PDF и сохраняет каждый в отдельный CSV."""
    os.makedirs(output_dir, exist_ok=True)
    out_files = []
    for i, pdf_path in enumerate(pdf_paths):
        out_csv = os.path.join(output_dir, f"raw_extracted_{i}.csv")
        rows = extract_pdf_to_rows(pdf_path)
        with open(out_csv, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerows(rows)
        print(f"  ✔ {pdf_path}  →  {out_csv}  ({len(rows)} строк)")
        out_files.append(out_csv)
    return out_files


PDF_FILES = [
    r"C:\Users\Asus\Downloads\gold_statement.pdf",   # ← замените на свои пути
    r"С:\Users\Asus\Downloads\gold_statement (2).pdf",
    r"С:\Users\Asus\Downloads\gold_statement (3).pdf",
    r"С:\Users\Asus\Downloads\Выписка.pdf"

]

# raw_csvs = extract_all_pdfs(PDF_FILES, output_dir="halyk_raw")
# print(f"\n✅ Извлечено файлов: {len(raw_csvs)}")

raw_csvs= pd.read_csv(r"C:\Users\Asus\OneDrive\Рабочий стол\projects\fmedia\final_extracted_table.csv", header=None)
print(f"✅ Загружено из CSV: {len(raw_csvs)} строк")


✅ Загружено из CSV: 4861 строк


In [12]:
import pandas as pd
import numpy as np
import re
import random
from datetime import datetime

# frames = []
# for csv_path in raw_csvs:
#     with open(csv_path, "r", encoding="utf-8") as f:
#         import csv as _csv
#         data = list(_csv.reader(f))
#     frames.append(pd.DataFrame(data))

# raw_df = pd.concat(frames, axis=0, ignore_index=True)
# print(f"Raw rows: {len(raw_df)}")
raw_df = raw_csvs

raw_df = raw_df.replace("", np.nan)
raw_df = raw_df.dropna(how="all")
raw_df = raw_df[raw_df.iloc[:, 0] != "Дата"]
raw_df = raw_df.reset_index(drop=True)

raw_df.columns = [str(c) for c in raw_df.columns]
col_count = len(raw_df.columns)
col_names = ["Дата", "Сумма", "Операция", "Детали"] + [f"extra_{i}" for i in range(max(0, col_count - 4))]
raw_df.columns = col_names[:col_count]

# Мерчант = extra_0 (если есть), иначе NaN
raw_df["Мерчант"] = raw_df["extra_0"] if "extra_0" in raw_df.columns else np.nan

# ── 2.4 Парсинг дат  (два формата: "30.05.2026" и "30.05.26") ───────────────
def parse_date(s):
    if pd.isna(s):
        return pd.NaT
    s = str(s).strip()
    for fmt in ("%d.%m.%Y", "%d.%m.%y"):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            pass
    return pd.NaT

raw_df["Дата"] = raw_df["Дата"].apply(parse_date)
raw_df = raw_df[raw_df["Дата"].notna()].copy()


def parse_amount(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()

    s = s.replace("₸", "").replace("\xa0", "").strip()
    s = re.sub(r"\s*([+\-])\s*", r"\1", s)  # "- 10" → "-10", "+ 5" → "+5"

    has_dot   = "." in s
    has_comma = "," in s

    if has_dot and has_comma:
        # Новый: "-10,346.78" → запятая-тысяч, точка-дес.
        s = s.replace(",", "")
    elif has_comma and not has_dot:
        # Старый: "-10 000,00" → пробел-тысяч, запятая-дес.
        s = s.replace(" ", "").replace(",", ".")
    else:
        # только цифры / только точка
        s = s.replace(" ", "").replace(",", "")

    # Убираем всё кроме цифр, точки, знака
    s = re.sub(r"[^\d\.\-\+]", "", s)

    # Защита от двойных знаков типа "-250230-500"
    match = re.match(r"^([+\-]?)(\d+\.?\d*)$", s)
    if match:
        return abs(float(match.group(1) + match.group(2)))
    return np.nan

raw_df["Сумма_число"] = raw_df["Сумма"].apply(parse_amount)

# ── 2.6 Тип операции (нормализация) ─────────────────────────────────────────
operation_map = {
    "Покупка"                     : "purchase",
    "Перевод"                     : "transfer",
    "Пополнение"                  : "topup",
    "Поступление со своего счета" : "self_transfer",
    "Разное"                      : "misc",
    "KZT"                         : "purchase",   # новый формат
    "Платеж"                      : "payment",
}
raw_df["operation_type"] = raw_df["Операция"].map(operation_map).fillna("other")
raw_df["direction"] = np.where(raw_df["Сумма_число"] >= 0, "credit", "debit")

# ── 2.7 Назначаем user_id (если ещё нет) ─────────────────────────────────────
random.seed(42)
user_ids = ["user_" + str(x) for x in [809,665,471,777,304,966,521,379,560,347]]
if "user_id" not in raw_df.columns:
    raw_df["user_id"] = [random.choice(user_ids) for _ in range(len(raw_df))]

# ── 2.8 Категории ────────────────────────────────────────────────────────────
categories = [
    "Продукты","Коммуналка","Аренда","Такси","Транспорт","Подписки","Электроника",
    "Рестораны","Медицина","Одежда","Авто","Красота","Развлечения","Путешествия",
    "Образование","Спорт","Кредиты","Ремонт","Питомцы","Связь",
    "Подарки","Дети","Страхование","Штрафы","Инвестиции",
]
if "category" not in raw_df.columns:
    raw_df["category"] = [random.choice(categories) for _ in range(len(raw_df))]

# ── 2.9 Итоговый датафрейм транзакций ────────────────────────────────────────
transactions = raw_df[[
    "Дата", "Сумма", "Сумма_число", "Операция", "operation_type",
    "direction", "Детали", "Мерчант", "user_id", "category",
]].rename(columns={
    "Дата"        : "date",
    "Сумма"       : "amount_raw",
    "Сумма_число" : "amount",
    "Операция"    : "operation_raw",
    "Детали"      : "details",
}).copy()

transactions["date"] = pd.to_datetime(transactions["date"])
transactions = transactions.reset_index(drop=True)
transactions.index.name = "transaction_id"

print(f"✅ Transactions после очистки: {len(transactions)} строк")
print(f"   Дата:  {transactions['date'].min().date()} → {transactions['date'].max().date()}")
print(f"   NaN в amount: {transactions['amount'].isna().sum()}")
print(transactions.head(3))


✅ Transactions после очистки: 4860 строк
   Дата:  2025-05-30 → 2026-05-30
   NaN в amount: 61
                     date    amount_raw    amount operation_raw  \
transaction_id                                                    
0              2026-05-30  -10,346.78 ₸  10346.78           KZT   
1              2026-05-30   -2,670.00 ₸   2670.00           KZT   
2              2026-05-29   -1,730.00 ₸   1730.00           KZT   

               operation_type direction            details  \
transaction_id                                               
0                    purchase    credit  Сумма в обработке   
1                    purchase    credit  Сумма в обработке   
2                    purchase    credit  Сумма в обработке   

                                             Мерчант   user_id category  
transaction_id                                                           
0               OPENAI CHATGPT SUBSCR 14158799686 US  user_665  Красота  
1                             GASTRO

In [14]:

tx = transactions.dropna(subset=["amount"]).copy()

# ── 3.1  stats: avg & median по user_id × category ──────────────────────────
stats = (
    tx.groupby(["user_id", "category"])["amount"]
    .agg(
        count="count",
        total="sum",
        avg="mean",
        median="median",
        min_amount="min",
        max_amount="max",
    )
    .reset_index()
)
stats[["total","avg","median","min_amount","max_amount"]] = (
    stats[["total","avg","median","min_amount","max_amount"]].round(2)
)

print("── stats (user_id × category) ──────────────────────────────────")
print(stats.head(10).to_string(index=False))

# ── 3.2  user_summary ────────────────────────────────────────────────────────
user_summary = (
    tx.groupby("user_id")["amount"]
    .agg(
        total_transactions="count",
        total_spent="sum",
        avg_transaction="mean",
        median_transaction="median",
    )
    .reset_index()
)
user_summary[["total_spent","avg_transaction","median_transaction"]] = (
    user_summary[["total_spent","avg_transaction","median_transaction"]].round(2)
)

print("\n── user_summary ────────────────────────────────────────────────")
print(user_summary.to_string(index=False))


── stats (user_id × category) ──────────────────────────────────
 user_id    category  count     total      avg  median  min_amount  max_amount
user_304        Авто     15  83859.40  5590.63  1709.4      160.00     49000.0
user_304      Аренда     17  97095.00  5711.47  1000.0       85.00     75500.0
user_304        Дети     17 101242.86  5955.46  2000.0       12.86     20000.0
user_304  Инвестиции     23 114327.00  4970.74  1850.0      100.00     32000.0
user_304  Коммуналка     20  60905.00  3045.25   895.0      100.00     22365.0
user_304     Красота     19  41901.60  2205.35  2000.0       11.60      7700.0
user_304     Кредиты     20 101968.00  5098.40  2245.0       80.00     44800.0
user_304    Медицина     16  92770.00  5798.12  1800.0       11.00     41000.0
user_304 Образование     28 121555.00  4341.25  1925.0       80.00     30000.0
user_304      Одежда     20 520047.00 26002.35 10467.0      120.00    200000.0

── user_summary ────────────────────────────────────────────────


In [15]:
# ── 6. EXPORT — сохраняем финальные CSV ─────────────────────────────────────
transactions.reset_index().to_csv("final_transactions.csv", index=False, encoding="utf-8-sig")
stats.to_csv("final_stats.csv", index=False, encoding="utf-8-sig")
user_summary.to_csv("final_user_summary.csv", index=False, encoding="utf-8-sig")

print("✅ Файлы сохранены:")
print("   • final_transactions.csv  — очищенные транзакции")
print("   • final_stats.csv         — avg & median по user_id × category")
print("   • final_user_summary.csv  — сводка по пользователям")
print("   • halyk_bank.db           — SQLite база данных (3 таблицы)")


✅ Файлы сохранены:
   • final_transactions.csv  — очищенные транзакции
   • final_stats.csv         — avg & median по user_id × category
   • final_user_summary.csv  — сводка по пользователям
   • halyk_bank.db           — SQLite база данных (3 таблицы)
